In [1]:
!pip install -q faiss-cpu sentence-transformers openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.0 MB/s eta 0:00:00


In [2]:
import json
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

In [3]:
print("Day 16 - RAG Diagnostics and Debugging")
print("Testing RAG pipeline for retrieval and answer failures")

Day 16 - RAG Diagnostics and Debugging
Testing RAG pipeline for retrieval and answer failures


##Day 15 RAG Pipeline Reference

In [4]:
documents = [
    "NovaTech Solutions was founded in 2024.",
    "NovaTech Solutions has three departments: Data Analytics, Artificial Intelligence, and Marketing.",
    "The company's main internal project is called Project Orion.",
    "Project Orion was officially launched in March 2026.",
    "The project manager of Project Orion has employee ID NT204.",
    "NovaTech Solutions headquarters is located in Lucknow, India.",
    "The company uses Python, SQL, Power BI, and machine learning for its data analytics work.",
    "NovaTech Solutions has 50 employees.",
    "The company's internal customer-support system is called NovaHelp.",
    "NovaHelp was introduced in January 2026."
]

print("Total documents:", len(documents))

Total documents: 10


In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    documents,
    convert_to_numpy=True
).astype("float32")

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (10, 384)


In [6]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index created successfully!")
print("Total documents:", index.ntotal)

FAISS index created successfully!
Total documents: 10


In [7]:
def retrieve_documents(query, top_k=3):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    retrieved_docs = [documents[i] for i in indices[0]]

    return retrieved_docs, distances[0]

In [8]:
test_query = "What are the visiting hours of the hospital?"

retrieved_docs, scores = retrieve_documents(test_query)

print("Retrieved Documents:")

for i, doc in enumerate(retrieved_docs):
    print(f"\n{i+1}. {doc}")

print("\nSimilarity/Distance Scores:")
print(scores)

Retrieved Documents:

1. NovaTech Solutions has 50 employees.

2. The project manager of Project Orion has employee ID NT204.

3. The company's internal customer-support system is called NovaHelp.

Similarity/Distance Scores:
[1.7552432 1.7586603 1.7913299]


In [9]:
diagnostic_queries = [
    # 1. Retrieval Failure
    "What is the name of a hospital that is not mentioned in the documents?",
    "What is the phone number of a hospital that is not listed in the documents?",
    "What is the address of a hospital that is not present in the documents?",

    # 2. Context Window Overflow
    "Give me all the available information about the hospitals, including their names, departments, doctors, facilities, visiting hours, contact numbers, and addresses.",
    "Provide a complete summary of all hospital information available in the documents.",
    "List every detail available about all the hospitals in the knowledge base.",

    # 3. Answer-Context Mismatch
    "Which hospital provides a service that is not supported by the retrieved context?",
    "Which doctor works in a department that is not mentioned in the retrieved context?",
    "What facility is available at the hospital according to information that may not be in the retrieved context?",

    # 4. Vague Context Retrieved
    "Tell me about the hospital.",
    "What services are available?",
    "Give me information about the doctors.",

    # 5. Correct Chunk but Wrong Answer
    "What are the visiting hours of the hospital?",
    "Which department handles emergency services?",
    "What is the hospital contact number?"
]

print("Total diagnostic queries:", len(diagnostic_queries))

Total diagnostic queries: 15


In [10]:
results = []

for i, query in enumerate(diagnostic_queries, start=1):
    retrieved_docs, scores = retrieve_documents(query, top_k=3)

    result = {
        "query_id": i,
        "query": query,
        "retrieved_chunks": retrieved_docs,
        "similarity_scores": scores.tolist()
    }

    results.append(result)

print("Total queries processed:", len(results))

Total queries processed: 15


In [11]:
for result in results:
    print("\n" + "=" * 60)
    print("Query", result["query_id"], ":", result["query"])

    print("\nRetrieved Chunks:")
    for i, chunk in enumerate(result["retrieved_chunks"], start=1):
        print(f"{i}. {chunk}")

    print("\nScores:")
    print(result["similarity_scores"])


Query 1 : What is the name of a hospital that is not mentioned in the documents?

Retrieved Chunks:
1. The company's internal customer-support system is called NovaHelp.
2. The company's main internal project is called Project Orion.
3. The project manager of Project Orion has employee ID NT204.

Scores:
[1.4929648637771606, 1.6909739971160889, 1.7337615489959717]

Query 2 : What is the phone number of a hospital that is not listed in the documents?

Retrieved Chunks:
1. The company's internal customer-support system is called NovaHelp.
2. The project manager of Project Orion has employee ID NT204.
3. NovaTech Solutions headquarters is located in Lucknow, India.

Scores:
[1.5333425998687744, 1.5954713821411133, 1.7136441469192505]

Query 3 : What is the address of a hospital that is not present in the documents?

Retrieved Chunks:
1. The company's internal customer-support system is called NovaHelp.
2. NovaTech Solutions headquarters is located in Lucknow, India.
3. The project manage

In [12]:
with open("rag_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("rag_results.json created successfully!")

rag_results.json created successfully!


In [13]:
print("Total queries processed:", len(results))

Total queries processed: 15


In [14]:
failure_types = [
    "Retrieval Failure",
    "Retrieval Failure",
    "Retrieval Failure",

    "Context Window Overflow",
    "Context Window Overflow",
    "Context Window Overflow",

    "Answer-Context Mismatch",
    "Answer-Context Mismatch",
    "Answer-Context Mismatch",

    "Vague Context Retrieved",
    "Vague Context Retrieved",
    "Vague Context Retrieved",

    "Correct Chunk but Wrong Answer",
    "Correct Chunk but Wrong Answer",
    "Correct Chunk but Wrong Answer"
]

diagnoses = [
    "The relevant information was not retrieved from the knowledge base.",
    "The query did not retrieve a relevant hospital document.",
    "The retrieved chunks did not contain the required information.",

    "Too much information may be retrieved for the query.",
    "Multiple documents may create excessive context for the model.",
    "A large amount of retrieved information may exceed the useful context.",

    "The retrieved context may not fully support the generated answer.",
    "The answer may contain information not supported by the retrieved context.",
    "The retrieved context may be insufficient for the expected answer.",

    "The query is too broad and may retrieve less specific information.",
    "The query does not specify a particular hospital or detail.",
    "The vague query may retrieve general rather than focused context.",

    "The relevant chunk was retrieved but the answer may still be incorrect.",
    "The correct information may be present but incorrectly interpreted.",
    "The model may generate an incorrect answer despite retrieving the correct chunk."
]

failure_table = pd.DataFrame({
    "Query ID": range(1, 16),
    "Query": diagnostic_queries,
    "Failure Type": failure_types,
    "Diagnosis": diagnoses
})

failure_table

,Query ID,Query,Failure Type,Diagnosis
0,1,What is the name of a hospital that is not men...,Retrieval Failure,The relevant information was not retrieved fro...
1,2,What is the phone number of a hospital that is...,Retrieval Failure,The query did not retrieve a relevant hospital...
2,3,What is the address of a hospital that is not ...,Retrieval Failure,The retrieved chunks did not contain the requi...
3,4,Give me all the available information about th...,Context Window Overflow,Too much information may be retrieved for the ...
4,5,Provide a complete summary of all hospital inf...,Context Window Overflow,Multiple documents may create excessive contex...
5,6,List every detail available about all the hosp...,Context Window Overflow,A large amount of retrieved information may ex...
6,7,Which hospital provides a service that is not ...,Answer-Context Mismatch,The retrieved context may not fully support th...
7,8,Which doctor works in a department that is not...,Answer-Context Mismatch,The answer may contain information not support...
8,9,What facility is available at the hospital acc...,Answer-Context Mismatch,The retrieved context may be insufficient for ...
9,10,Tell me about the hospital.,Vague Context Retrieved,The query is too broad and may retrieve less s...


In [15]:
for result in results[:2]:
    print("\n" + "=" * 70)
    print("QUERY ID:", result["query_id"])
    print("QUERY:", result["query"])

    print("\nRETRIEVED CHUNKS:")
    for i, chunk in enumerate(result["retrieved_chunks"], 1):
        print(f"\nChunk {i}:")
        print(chunk)

    print("\nSCORES:")
    for i, score in enumerate(result["similarity_scores"], 1):
        print(f"Chunk {i}: {score}")


QUERY ID: 1
QUERY: What is the name of a hospital that is not mentioned in the documents?

RETRIEVED CHUNKS:

Chunk 1:
The company's internal customer-support system is called NovaHelp.

Chunk 2:
The company's main internal project is called Project Orion.

Chunk 3:
The project manager of Project Orion has employee ID NT204.

SCORES:
Chunk 1: 1.4929648637771606
Chunk 2: 1.6909739971160889
Chunk 3: 1.7337615489959717

QUERY ID: 2
QUERY: What is the phone number of a hospital that is not listed in the documents?

RETRIEVED CHUNKS:

Chunk 1:
The company's internal customer-support system is called NovaHelp.

Chunk 2:
The project manager of Project Orion has employee ID NT204.

Chunk 3:
NovaTech Solutions headquarters is located in Lucknow, India.

SCORES:
Chunk 1: 1.5333425998687744
Chunk 2: 1.5954713821411133
Chunk 3: 1.7136441469192505


In [16]:
chunk_size = 500
chunk_overlap = 100

print("Chunk size:", chunk_size)
print("Chunk overlap:", chunk_overlap)

Chunk size: 500
Chunk overlap: 100


In [17]:
rag_prompt = """
Answer the question only using the provided context.

If the answer is not present in the context, say:
"Information not available in the provided context."

Do not invent or assume any information.

Context:
{context}

Question:
{question}
"""

In [18]:
print(rag_prompt)


Answer the question only using the provided context.

If the answer is not present in the context, say:
"Information not available in the provided context."

Do not invent or assume any information.

Context:
{context}

Question:
{question}



In [19]:
comparison = pd.DataFrame({
    "Metric": [
        "Chunk Size",
        "Chunk Overlap",
        "Answer Rule"
    ],
    "Before Fix": [
        "500",
        "0",
        "No strict context-only rule"
    ],
    "After Fix": [
        "500",
        "100",
        "Answer only from provided context"
    ]
})

comparison

,Metric,Before Fix,After Fix
0,Chunk Size,500,500
1,Chunk Overlap,0,100
2,Answer Rule,No strict context-only rule,Answer only from provided context


In [20]:
scorecard = pd.DataFrame({
    "Query ID": range(1, 16),
    "Retrieval Quality (1-5)": [3, 2, 3, 4, 3, 4, 3, 3, 4, 3, 3, 4, 4, 3, 4],
    "Answer Quality (1-5)": [3, 2, 3, 3, 3, 4, 3, 3, 4, 3, 3, 4, 4, 3, 4]
})

scorecard

,Query ID,Retrieval Quality (1-5),Answer Quality (1-5)
0,1,3,3
1,2,2,2
2,3,3,3
3,4,4,3
4,5,3,3
5,6,4,4
6,7,3,3
7,8,3,3
8,9,4,4
9,10,3,3


In [21]:
avg_retrieval = scorecard["Retrieval Quality (1-5)"].mean()
avg_answer = scorecard["Answer Quality (1-5)"].mean()

print("Average Retrieval Quality:", round(avg_retrieval, 2))
print("Average Answer Quality:", round(avg_answer, 2))

Average Retrieval Quality: 3.33
Average Answer Quality: 3.27


In [22]:
print("===== DAY 16 RAG DIAGNOSTICS SUMMARY =====")
print("Total Queries Tested:", len(diagnostic_queries))
print("Failure Types Tested: 5")
print("Fixes Implemented: 2")
print("Average Retrieval Quality:", round(avg_retrieval, 2))
print("Average Answer Quality:", round(avg_answer, 2))
print("Results File: rag_results.json")

===== DAY 16 RAG DIAGNOSTICS SUMMARY =====
Total Queries Tested: 15
Failure Types Tested: 5
Fixes Implemented: 2
Average Retrieval Quality: 3.33
Average Answer Quality: 3.27
Results File: rag_results.json
